In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
branch_table = dbutils.widgets.get("branch_table")
patient_table = dbutils.widgets.get("patient_table")
patientpayer_table = dbutils.widgets.get("patientpayer_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW atb_src AS
SELECT 
  CAST(ReportingDate AS DATE) AS ReportingDate, 
  CAST(FacilityCode AS INT) AS FacilityCode, 
  CAST(AcctNbr AS STRING) AS AcctNbr, 
  CAST(AcctBalance AS DOUBLE) AS AcctBalance, 
  CAST(SourceSystemKey AS INT) AS SourceSystemKey
FROM (
  WITH 
  atb_cte AS (
    SELECT
      b.ExternalId AS FacilityCode,
      bl.ClaimNumber AS AcctNbr,
      CAST('{fetch_date}' AS DATE) AS ReportingDate,
      bl.AmountBilled AS AcctBalance,
      '19' AS SourceSystemKey
    FROM {source_table} bl
    left JOIN {patientpayer_table} pp on pp.id=bl.PatientPayerId
    left JOIN {patient_table} p on p.id=pp.PatientId
    LEFT JOIN {branch_table} b ON b.Id = p.BranchId  -- Join branch to get the real facility code
    WHERE bl.isActive='true'
  ),
  atb_clean AS (
    SELECT *,
    row_number() OVER (PARTITION BY AcctNbr ORDER BY AcctNbr ) AS rn
    FROM atb_cte
  )
  SELECT ReportingDate, FacilityCode, AcctNbr, AcctBalance, SourceSystemKey
  FROM atb_clean
  WHERE rn=1
)
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING atb_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 19

WHEN MATCHED THEN
  UPDATE SET
    tgt.FacilityCode = src.FacilityCode,
    tgt.AcctBalance = src.AcctBalance,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
  INSERT (
    ReportingDate,
    FacilityCode,
    AcctNbr,
    AcctBalance,
    SourceSystemKey,
    _load_timestamp
  )
  VALUES (
    src.ReportingDate,
    src.FacilityCode,
    src.AcctNbr,
    src.AcctBalance,
    src.SourceSystemKey,
    current_timestamp()
  );
""")
)